# Main

In [6]:
# importing packages
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

In [7]:

# -------------------------------------------------
# 2️⃣  Define absolute paths to the three source files
# -------------------------------------------------
ROOT = "/Users/marierobillon/LocalFiles/DS_causal_inf_for_sustainability_teamwork"
RAW = f"{ROOT}/data/data/raw"
ELCOM_CSV   = f"{RAW}/elcom_tariffs_H2_H4_H7.csv"
BFE_PLANTS  = f"{RAW}/bfe_plants.csv/ElectricityProductionPlant.csv"
DWELLINGS   = f"{RAW}/bfs_dwellings_by_occupant_2000_all.csv"
# -------------------------------------------------

# Data

## Loading

### X variable

In [ ]:

# Load **X** – ElCom household tariffs
# -------------------------------------------------
# The file uses ';' as separator; we only need the H4 total column and the
# identifiers that let us map to communes and years.
elcom_raw = pd.read_csv(ELCOM_CSV, sep=",", low_memory=False)
# Standardise column names (lower‑case, underscores)
elcom_raw.columns = (
    elcom_raw.columns.str.strip()
                .str.lower()
                .str.replace(" ", "_")
                .str.replace(r"[^\w]", "", regex=True)
)
# Relevant columns (adjust if the actual names differ):
#   - commune_number  : old commune ID
#   - year            : tariff year
#   - grid_operator   : operator code (we will average later)
#   - h4_total        : total H4 tariff (ct/kWh)
#   - h4_energy       : energy component of H4 (optional for later)

print("🔎 Column names in the file:")
for col in elcom_raw.columns:
    print(f" • {col}")

#cols_needed = ["commune_number", "year", "grid_operator", "h4_total", "h4_energy"]
#elcom = elcom[cols_needed]


🔎 Column names in the file:
 • period
 • municipality
 • operator
 • category
 • total
 • energy
 • grid
 • charge
 • aidfee
 • fixcosts


In [ ]:
# Keep only the columns we need for X
# -----------------------------------
cols_needed = ["municipality", "period", "operator", "category", "total", "energy"]
elcom = elcom_raw[cols_needed].copy()
# Rename to the semantic names we’ll use later
elcom = elcom.rename(columns={
    "municipality":   "commune_number",   # old commune ID
    "period":         "year",             # tariff year (int)
    "operator":       "grid_operator",    # operator code
    "category":       "tariff_profile",   # H2 / H4 / H7
    "total":          "total_ct_per_kwh", # total H4 tariff (cent/kWh)
    "energy":         "energy_ct_per_kwh" # energy component (cent/kWh)
})

print(elcom.head())

   commune_number  year  grid_operator tariff_profile  total_ct_per_kwh  \
0            3921  2018             10             H2            25.359   
1            4045  2018            804             H2            17.801   
2            5590  2018            698             H2            25.480   
3             979  2018            559             H2            22.802   
4            4080  2018            594             H2            20.678   

   energy_ct_per_kwh  
0              7.765  
1              4.264  
2              8.900  
3              6.763  
4              4.798  


In [21]:
# -------------------------------------------------
# Filter to the H4 profile (4 500 kWh / yr – the standard benchmark for a typical Swiss household.)
# -------------------------------------------------
elcom_h4 = elcom[elcom["tariff_profile"] == "H4"].copy()

In [ ]:
# -------------------------------------------------
# 4️⃣  Load **Y** – SFOE plant register (solar plants only)
# -------------------------------------------------
plants = pd.read_csv(BFE_PLANTS, low_memory=False)
# Clean column names
plants.columns = (
    plants.columns.str.strip()
                .str.lower()
                .str.replace(" ", "_")
                .str.replace(r"[^\w]", "", regex=True)
)
# Keep only solar plants (SubCategory code = 'subcat_2')
# We need: commune_number (or postcode → commune), commissioning year,
# capacity (kW), and coordinates (_x, _y) for spatial checks.
solar = plants[plants["subcategory"] == "subcat_2"].copy()
# Extract year from commissioning date (format: YYYY‑MM‑DD or YYYY)
solar["commission_year"] = pd.to_datetime(solar["commissioning_date"], errors="coerce").dt.year
cols_needed_y = ["commune_number", "commission_year", "capacity_kw", "_x", "_y"]
solar = solar[cols_needed_y]

In [ ]:
# -------------------------------------------------
# 5️⃣  Load **Z** – Census 2000 dwellings by occupant
# -------------------------------------------------
dwell = pd.read_csv(DWELLINGS, low_memory=False)
# Clean column names
dwell.columns = (
    dwell.columns.str.strip()
               .str.lower()
               .str.replace(" ", "_")
               .str.replace(r"[^\w]", "", regex=True)
)
# Keep the total dwellings and the owner‑occupied count.
# The exact column names may differ; adjust if needed.
# Typical names in the Swiss census file:
#   - bfs_nr            : commune ID
#   - total_dwellings   : all dwellings
#   - owner_occupied    : owner‑occupied dwellings
cols_needed_z = ["bfs_nr", "total_dwellings", "owner_occupied"]
dwell = dwell[cols_needed_z]

In [ ]:
# -------------------------------------------------
# 6️⃣  Helper: map old commune numbers → 2026 numbers
# -------------------------------------------------
# The cross‑walk is usually stored in a separate CSV (e.g. bfs_commune_levels_2025.csv).
# If you have such a file, load it here; otherwise skip this step.
# Example (uncomment & adjust path if the file exists):
# CROSSWALK = f"{RAW}/bfs_commune_levels_2025.csv"
# cross = pd.read_csv(CROSSWALK, low_memory=False)
# cross.columns = (
#     cross.columns.str.strip()
#                .str.lower()
#                .str.replace(" ", "_")
#                .str.replace(r"[^\w]", "", regex=True)
# )
# # Assume columns: old_commune_number → new_commune_number_2026
# cross = cross[["old_commune_number", "new_commune_number_2026"]]
# # Join to each table where needed
# elcom = elcom.merge(cross, left_on="commune_number", right_on="old_commune_number", how="left")
# solar = solar.merge(cross, left_on="commune_number", right_on="old_commune_number", how="left")
# dwell = dwell.merge(cross, left_on="bfs_nr", right_on="old_commune_number", how="left")
# # Replace old IDs with the new ones
# for df in [elcom, solar, dwell]:
#     df["commune_id"] = df["new_commune_number_2026"].fillna(df["commune_number"])
#     df.drop(columns=["commune_number", "old_commune_number", "new_commune_number_2026"], inplace=True)
# If you don’t have a cross‑walk file, just use the original IDs:
elcom["commune_id"] = elcom["commune_number"]
solar["commune_id"] = solar["commune_number"]
dwell["commune_id"] = dwell["bfs_nr"]
# -------------------------------------------------
# 7️⃣  Compute **X** – Change in H4 total tariff (2023 – 2022)
# -------------------------------------------------
# First average tariff across operators for each commune‑year.
x_avg = (
    elcom.groupby(["commune_id", "year"])["h4_total"]
         .mean()
         .reset_index(name="h4_total_avg")
)
# Pivot to have 2022 and 2023 side‑by‑side
x_pivot = x_avg.pivot(index="commune_id", columns="year", values="h4_total_avg")
# Keep only 2022 & 2023 (if missing, result will be NaN)
x_pivot = x_pivot[[2022, 2023]] if {2022, 2023}.issubset(x_pivot.columns) else x_pivot
# X = 2023 – 2022
x_pivot["X"] = x_pivot[2023] - x_pivot[2022]
X = x_pivot["X"].rename("X")          # Series indexed by commune_id
# -------------------------------------------------
# 8️⃣  Compute **Y** – New solar installations per commune‑year
# -------------------------------------------------
y_counts = (
    solar.groupby(["commune_id", "commission_year"])
         .size()
         .reset_index(name="new_solar_plants")
)
# If you also want total capacity per year:
y_capacity = (
    solar.groupby(["commune_id", "commission_year"])["capacity_kw"]
         .sum()
         .reset_index(name="solar_capacity_kw")
)
Y = y_counts.pivot(index="commune_id", columns="commission_year", values="new_solar_plants")
# Optional: keep capacity side‑by‑side as a second DataFrame
Y_capacity = y_capacity.pivot(index="commune_id", columns="commission_year", values="solar_capacity_kw")
# -------------------------------------------------
# 9️⃣  Compute **Z** – Owner‑occupied share of dwellings
# -------------------------------------------------
dwell["Z"] = dwell["owner_occupied"] / dwell["total_dwellings"]
Z = dwell.set_index("commune_id")["Z"]
# -------------------------------------------------
# 10️⃣  Assemble a tidy DataFrame with X, Y, Z side‑by‑side
# -------------------------------------------------
final = pd.concat([X, Z], axis=1)               # X and Z are Series
# For Y we keep the yearly counts; you can also collapse to a single metric.
# Example: total new solar plants from 2020‑2026
if Y is not None:
    y_total = Y.loc[:, 2020:2026].sum(axis=1).rename("Y_total_new_solar_2020_2026")
    final = final.join(y_total, how="left")
print("✅ Finished loading X, Y, Z")
display(final.head())

## Data visualisation

### PV plant commissioning dates, capacity, and locations

In [9]:
# Loading data
data_dir = Path.cwd() / "data/ch.bfe.elektrizitaetsproduktionsanlagen"
power_plants_path = data_dir / "ElectricityProductionPlant.csv" 
PV_details_path = data_dir /"PlantDetail.csv"
power_plants_df = pd.read_csv(power_plants_path)
PV_details_df = pd.read_csv(PV_details_path)

In [10]:

#Taking a look at the powerplants
print(power_plants_df.head()) 

   xtf_id      Address  PostCode          Municipality Canton       EGID  \
0    5686   Binenweg 5      3904                Naters     VS   892091.0   
1    5726     Robbia 2      7741  S. Carlo (Poschiavo)     GR  9080323.0   
2    5727          NaN      7744          Campocologno     GR        NaN   
3    5730      Büdemji      7240                Küblis     GR        NaN   
4    5740  Auenstrasse      8783               Linthal     GL        NaN   

  BeginningOfOperation  InitialPower  TotalPower MainCategory SubCategory  \
0           1969-09-01      349576.0    349576.0    maincat_1    subcat_1   
1           2024-01-16       38250.0     38250.0    maincat_1    subcat_1   
2           1907-03-01       55000.0     55000.0    maincat_1    subcat_1   
3           1922-01-01       44200.0     44200.0    maincat_1    subcat_1   
4           1962-03-18     1526000.0   1526000.0    maincat_1    subcat_1   

  PlantCategory           _x           _y  
0    plantcat_7  2642480.025  113023

**What does all these categories mean?**

| Category | ID          | English description |
|----------|------------|----------------------|
| **Main** | maincat_1   | Hydroelectric power |
| **Main** | maincat_2   | Other renewable energies |
| **Main** | maincat_3   | Nuclear energy |
| **Main** | maincat_4   | Fossil fuel |
| **Sub**  | subcat_1    | Hydroelectric power |
| **Sub**  | subcat_2    | Photovoltaic |
| **Sub**  | subcat_3    | Wind energy |
| **Sub**  | subcat_4    | Biomass |
| **Sub**  | subcat_5    | Geothermal energy |
| **Sub**  | subcat_6    | Nuclear energy |
| **Sub**  | subcat_7    | Crude oil |
| **Sub**  | subcat_8    | Natural gas |
| **Sub**  | subcat_9    | Coal |
| **Sub**  | subcat_10   | Waste |
| **Plant**| plantcat_1  | Wastewater power plant |
| **Plant**| plantcat_2  | Diversion power plant |
| **Plant**| plantcat_3  | Weir plant |
| **Plant**| plantcat_4  | Continuous power plant |
| **Plant**| plantcat_5  | Drinking water power plant |
| **Plant**| plantcat_6  | Pumped storage power plant |
| **Plant**| plantcat_7  | Storage power plant |
| **Plant**| plantcat_8  | Attached |
| **Plant**| plantcat_9  | Integrated |
| **Plant**| plantcat_10 | Freestanding |
| **Plant**| plantcat_11 | Biomass usage |
| **Plant**| plantcat_12 | Waste incineration |
| **Plant**| plantcat_13 | Wastewater treatment |

In [ ]:
# selecting only the PV plants
PV_plants_df = power_plants_df[power_plants_df["SubCategory"] == "subcat_2"]
print(PV_plants_df.size)
print(power_plants_df.size)


4724370
4752986


In [34]:
#Taking a look at the pv plant details
print(PV_details_df.head()) 

   xtf_id        Date   Power  Inclination Orientation PlantCategory  \
0       1  2007-12-20  572.30          5.0    orient_4    plantcat_8   
1       2  2008-04-23    7.92         30.0    orient_7    plantcat_8   
2       3  2007-09-05    2.45         25.0    orient_1    plantcat_8   
3       4  2007-07-20    2.58          5.0    orient_5    plantcat_8   
4       5  2008-07-02    8.95         32.0    orient_5    plantcat_8   

   ElectricityProductionPlantR  
0                         6153  
1                         9482  
2                         9789  
3                        10396  
4                        10908  


**What does all these categories mean?**

| Category | ID          | English description |
|----------|------------|----------------------|
| **Plant**| plantcat_1  | Wastewater power plant |
| **Plant**| plantcat_2  | Diversion power plant |
| **Plant**| plantcat_3  | Weir plant |
| **Plant**| plantcat_4  | Continuous power plant |
| **Plant**| plantcat_5  | Drinking water power plant |
| **Plant**| plantcat_6  | Pumped storage power plant |
| **Plant**| plantcat_7  | Storage power plant |
| **Plant**| plantcat_8  | Attached |
| **Plant**| plantcat_9  | Integrated |
| **Plant**| plantcat_10 | Freestanding |
| **Plant**| plantcat_11 | Biomass usage |
| **Plant**| plantcat_12 | Waste incineration |
| **Plant**| plantcat_13 | Wastewater treatment |
| **Orientation**| orient_1 | North |
| **Orientation**| orient_2 | Northeast |
| **Orientation**| orient_3 | East |
| **Orientation**| orient_4 | Southeast |
| **Orientation**| orient_5 | South |
| **Orientation**| orient_6 | Southwest |
| **Orientation**| orient_7 | West |
| **Orientation**| orient_8 | Northwest |
| **Orientation**| orient_9 | Zenith |

You can join them on ElectricityProductionPlant.xtf_id = PlantDetail.ElectricityProductionPlantR to combine static plant metadata with its temporal performance data.

In [17]:
# Merge the PV plant metadata (in PV_plants_df) with their temporal performance records (in PV_details_df)
#   – `xtf_id`  : primary key in the plant table  
#   – `ElectricityProductionPlantR` : foreign key in the detail table
# Use an inner join so we retain only rows that have a matching measurement.

PV_merged_df = PV_plants_df.merge(
    PV_details_df,
    left_on="xtf_id",                     # column in PV_plants_df
    right_on="ElectricityProductionPlantR",# column in PV_details_df
    how="inner"
)

# Sanity check
print(f"Total plants loaded          : {len(power_plants_df):,}")
print(f"PV plants (subcat_2)         : {len(PV_plants_df):,}")
print(f"PV‑detail rows after merge   : {len(PV_merged_df):,}")

# Show the first few rows of the merged DataFrame
PV_merged_df.head()

Total plants loaded          : 339,499
PV plants (subcat_2)         : 337,455
PV‑detail rows after merge   : 406,676


,xtf_id_x,Address,PostCode,Municipality,Canton,EGID,BeginningOfOperation,InitialPower,TotalPower,MainCategory,...,PlantCategory_x,_x,_y,xtf_id_y,Date,Power,Inclination,Orientation,PlantCategory_y,ElectricityProductionPlantR
0,6153,Chemin du Champ-des-Filles 36a,1228,Plan-les-Ouates,GE,295111373.0,2007-12-20,572.30,572.30,maincat_2,...,plantcat_8,2496890.919,1113467.012,1,2007-12-20,572.30,5.0,orient_4,plantcat_8,6153
1,9482,Dorfmattweg 30,3110,Münsingen,BE,1358054.0,2008-04-23,7.92,7.92,maincat_2,...,plantcat_8,2608803.968,1191301.874,2,2008-04-23,7.92,30.0,orient_7,plantcat_8,9482
2,9789,Under Neuhus 6,6017,Ruswil,LU,2070412.0,2007-09-05,2.45,2.45,maincat_2,...,plantcat_8,2651282.402,1215289.458,3,2007-09-05,2.45,25.0,orient_1,plantcat_8,9789
3,10396,Hübel 4b,5034,Suhr,AG,101053497.0,2007-07-20,2.58,5.16,maincat_2,...,plantcat_8,2648047.030,1247249.394,4,2007-07-20,2.58,5.0,orient_5,plantcat_8,10396
4,10908,Steinertalweg 13,6422,Steinen,SZ,101235022.0,2008-07-02,7.03,8.95,maincat_2,...,plantcat_8,2689338.210,1211980.528,5,2008-07-02,8.95,32.0,orient_5,plantcat_8,10908


xtf_id_x → left DataFrame → PV_plants_df (plant‑level id) / xtf_id_y → right DataFrame → PV_details_df (measurement‑level id).